Downloading libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

Loading data

In [ ]:
# for Google Colab
'''
from google.colab import drive
drive.mount('/content/drive')

our_df = pd.read_excel('/content/drive/My Drive/result_set ML тест.xlsx', engine='openpyxl')
'''

In [ ]:
# for jupyter notebook

In [ ]:
our_df = pd.read_excel('result_set ML тест.xlsx', engine='openpyxl')

In [ ]:
our_df.head(10)

EDA

In [ ]:
our_df.shape

In [ ]:
our_df.dtypes

In [ ]:
our_df.describe().round(2)

In [ ]:
numeric_cols = our_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('mark')

In [ ]:
numeric_cols

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < 16:
        amplitude = our_df[col].max() - our_df[col].min()
        axes[i].hist(our_df[col].dropna(), bins=20, edgecolor='black')
        axes[i].set_title(f'{col}\nAmplitude: {amplitude:.2f}')

plt.tight_layout()
plt.show()

In [ ]:
for col in numeric_cols:
    Q1 = our_df[col].quantile(0.25)
    Q3 = our_df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = our_df[(our_df[col] < Q1 - 1.5*IQR) | (our_df[col] > Q3 + 1.5*IQR)]
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(our_df)*100:.1f}%)")

In [ ]:
print(our_df['Ft 12'].unique()[:20])
print(our_df['Ft 13'].unique()[:20])
print(our_df['Ft14'].unique()[:20])

In [ ]:
print(our_df['Ft 12'].value_counts().head(10))
print(our_df['Ft 13'].value_counts().head(10))
print(our_df['Ft14'].value_counts().head(10))

In [ ]:
our_df[(our_df['Ft 12'] == -1000) | (our_df['Ft 13'] == -1000)]

In [ ]:
our_df.shape

Features FT12 and FT13 each have two rows with outliers, likely due to a technical error. Their proportion is negligible. It's safe to remove the two rows where both features show outliers

In [ ]:
our_df = our_df[(our_df['Ft 12'] != -1000) & (our_df['Ft 13'] != -1000)]

In [ ]:
our_df.shape

In [ ]:
for col in numeric_cols:
    Q1 = our_df[col].quantile(0.25)
    Q3 = our_df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = our_df[(our_df[col] < Q1 - 1.5*IQR) | (our_df[col] > Q3 + 1.5*IQR)]
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(our_df)*100:.1f}%)")

In [ ]:
print(our_df['Ft 12'].unique()[:20])
print(our_df['Ft 13'].unique()[:20])
print(our_df['Ft14'].unique()[:20])

In [ ]:
real_numeric = ['Ft 0', 'Ft 1', 'Ft 2', 'Ft 3', 'Ft 4', 'Ft 5', 'Ft 6', 'Ft 7', 'Ft 10']

print("="*50)
print("Correlation")
print("="*50)

if len(real_numeric) > 1:
    correlation_matrix = our_df[real_numeric + ['mark']].corr()

    corr_with_mark = correlation_matrix['mark'].sort_values(ascending=False)
    print("Corr with mark:")
    print(corr_with_mark.round(3))

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title('Correlation matrix for numerics')
    plt.tight_layout()
    plt.show()

In [ ]:
# duplicates ?
print("duplicates:", our_df.duplicated().sum())

Let's take a look at NaNs

In [ ]:
nans = our_df.isnull().sum()

In [ ]:
nans

In features Ft10 and Ft11, about 1% of the values are missing, while in Ft15, about 16% are  missing.
 Ft10 is numeric -> let's fill with median; Ft11 is binary, let's fill Nans with mode
Ft 15 - the most important one. It's categoric. Let's fill NaNs with the most popular category

In [ ]:
# Filling
our_df['Ft 10'] = our_df['Ft 10'].fillna(our_df['Ft 10'].median())
our_df['Ft 11'] = our_df['Ft 11'].fillna(our_df['Ft 11'].mode()[0])
our_df['Ft15'] = our_df['Ft15'].fillna('unknown')

# check
print(our_df[['Ft 10', 'Ft 11', 'Ft15']].isna().sum())

Let's deal with the categorical features. They need to be processed

In [ ]:
cats = ['Ft 8', 'Ft 9', 'Ft 11', 'Ft 12', 'Ft 13', 'Ft14', 'Ft15', 'Ft16']

for col in cats:
    print(f"{col}: {sorted(our_df[col].dropna().unique())}")

Ft 8, Ft 11, Ft 16 --> one-hot encoding

Ft 15: added "unknown' category    so   -> one-hot encoding (label encoding is problematic)

Ft 9: only 4 values. OHE is possible but loses order - > use label encoding.
    
Ft 12 and Ft 13: unclear if ordinal or codes. We don't know -> leave as is

In [ ]:
from sklearn.preprocessing import LabelEncoder

# OHE for  Ft8 Ft11 Ft15 Ft16
ohe_cols = ['Ft 8', 'Ft 11', 'Ft15', 'Ft16']
ohe_df = pd.get_dummies(our_df[ohe_cols], drop_first=False)

# Label encoding for Ft 9
le = LabelEncoder()
ft9_encoded = le.fit_transform(our_df['Ft 9'])

final_df = pd.concat([
    our_df[['Ft 0', 'Ft 1', 'Ft 2', 'Ft 3', 'Ft 4', 'Ft 5', 'Ft 6', 'Ft 7', 'Ft 10', 'Ft 12', 'Ft 13', 'Ft14', 'Ft17', 'mark']],
    ohe_df,
    pd.Series(ft9_encoded, name='Ft9_encoded')
], axis=1)

print(f"Original shape: {our_df.shape}")
print(f"Final shape: {final_df.shape}")

For our understanding and checking let's compare non encoded our_df and encoded final_df

In [ ]:
our_df.head(5)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# All columns
final_df.head(5)

Let's check class disbalance

In [ ]:
print("="*50)
print("CLASS BALANCE CHECK")
print("="*50)

class_counts = our_df['mark'].value_counts()
class_percent = our_df['mark'].value_counts(normalize=True) * 100

balance_report = pd.DataFrame({
    'Count': class_counts,
    'Percentage': class_percent.round(2)
})
print(balance_report)

sns.set_style("darkgrid")
sns.set_palette(["#4A90E2", "#E24A4A"])

fig, ax = plt.subplots(figsize=(5, 5))

wedges, texts, autotexts = ax.pie(class_counts.values,
                                    labels=None,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 12, 'fontweight': 'bold'},
                                    pctdistance=0.75)

labels = [f'Class 0\n{class_counts[0]} ({class_percent[0]:.1f}%)',
          f'Class 1\n{class_counts[1]} ({class_percent[1]:.1f}%)']
ax.legend(wedges, labels, title="Target", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

ax.set_title('Target Variable Distribution', fontsize=14, pad=20)

ratio = class_percent[1] / class_percent[0]
print(f"\nClass ratio (1/0): {ratio:.3f}")
if ratio < 0.3:
    print("Warning: Significant class imbalance detected")
else:
    print("OK: Reasonable class balance")

plt.tight_layout()
plt.show()

We have a serious class imbalance. This will need to be taken into account during modeling and when choosing metrics

MODELS

Let's start with basic model - Logistic model

there are 4 rows with NaNs, let's clean them first

In [ ]:
final_df = final_df.dropna()
print(f"rows left: {len(final_df)}")

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, recall_score, precision_score, f1_score

X = final_df.drop('mark', axis=1)
y = final_df['mark']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

param_grid = {'lr__C': [0.001, 0.003, 0.01, 0.03, 0.1, 1, 5, 10], 'lr__penalty': ['l1', 'l2'], 'lr__solver': ['liblinear']}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(lr_pipeline, param_grid, cv=skf, scoring='average_precision', n_jobs=-1)
grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)
y_proba = grid_search.predict_proba(X_test)[:, 1]

print(f"Best parameters: {grid_search.best_params_}")
print(f"Test PR-AUC: {average_precision_score(y_test, y_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}, Recall: {recall_score(y_test, y_pred):.4f}, F1: {f1_score(y_test, y_pred):.4f}")

We have low precision rate, results are modest

In [ ]:
import sys
# !{sys.executable} -m pip install optuna
# !pip install xgboost

# !brew install libomp


In [ ]:
#  object to int for XGBoost
for col in final_df.columns:
    if final_df[col].dtype == 'object':
        final_df[col] = final_df[col].astype(int)

# check
print(final_df.dtypes.value_counts())
print(f"Object columns left: {final_df.select_dtypes(include=['object']).columns.tolist()}")

#  bool -> int
bool_cols = final_df.select_dtypes(include=['bool']).columns
for col in bool_cols:
    final_df[col] = final_df[col].astype(int)

In [ ]:
!pip install optuna


import xgboost as xgb
from sklearn.model_selection import cross_val_score
import optuna
from optuna.samplers import TPESampler

X = final_df.drop('mark', axis=1)
y = final_df['mark']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

print("="*50)
print("XGBOOST - OPTUNA")
print("="*50)
print(f"Scale pos weight: {scale_pos_weight:.2f}")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'aucpr',
        'random_state': 42
    }

    xgb_model = xgb.XGBClassifier(**params, use_label_encoder=False, verbosity=0)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(xgb_model, X_train, y_train, cv=skf, scoring='average_precision')
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest params: {study.best_params}")
print(f"Best CV PR-AUC: {study.best_value:.4f}")

best_xgb = xgb.XGBClassifier(
    **study.best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False,
    verbosity=0
)
best_xgb.fit(X_train, y_train)

y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
print(f"Test PR-AUC: {average_precision_score(y_test, y_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")

importances = pd.DataFrame({'feature': X.columns, 'importance': best_xgb.feature_importances_})
print(f"\nTop 10 features:\n{importances.sort_values('importance', ascending=False).head(10)}")

Results now are better, but let's try to improve them. Let's do something with class disbalance

In [ ]:
from imblearn.over_sampling import SMOTE

print("="*50)
print("XGBOOST WITH SMOTE")
print("="*50)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Class 0: {sum(y_train==0)}, Class 1: {sum(y_train==1)}")
print(f"After SMOTE - Class 0: {sum(y_train_bal==0)}, Class 1: {sum(y_train_bal==1)}")

# Train best_xgb on balanced data
best_xgb.fit(X_train_bal, y_train_bal)

# Predict
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("TEST RESULTS WITH SMOTE")
print("="*50)
print(f"Test PR-AUC: {average_precision_score(y_test, y_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")

SMOTE made the results worse. Let's try changing the threshold

In [ ]:
print("="*50)
print("THRESHOLD TUNING")
print("="*50)

y_proba = best_xgb.predict_proba(X_test)[:, 1]

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred = (y_proba >= threshold).astype(int)
    if y_pred.sum() > 0:
        p = precision_score(y_test, y_pred)
        r = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        print(f"Threshold {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f1:.3f}")
    else:
        print(f"Threshold {threshold}: No predictions")

# Best threshold (based on F1)
best_threshold = 0.35   
y_pred_best = (y_proba >= best_threshold).astype(int)
print(f"\nBest F1 with threshold {best_threshold}: {f1_score(y_test, y_pred_best):.3f}")

In [ ]:
# final model with threshold 0.35

y_proba = best_xgb.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba >= 0.35).astype(int)

print("FINAL MODEL (threshold=0.35):")
print(f"Precision: {precision_score(y_test, y_pred_final):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_final):.3f}")
print(f"F1: {f1_score(y_test, y_pred_final):.3f}")

In [ ]:
# So :
# FINAL MODEL (threshold=0.35):
# Precision: 0.450
# Recall: 0.692
# F1: 0.545

# let's try other models

In [ ]:
# lightGBM results
'''

==================================================
lgbm RESULTS
==================================================
Test PR-AUC: 0.3850
Threshold 0.3: P=0.500, R=0.077, F1=0.133
Threshold 0.35: P=0.500, R=0.077, F1=0.133
Threshold 0.4: P=0.500, R=0.077, F1=0.133
Threshold 0.45: P=0.500, R=0.077, F1=0.133
Threshold 0.5: P=0.500, R=0.077, F1=0.133
'''

In [ ]:
!pip install catboost


from catboost import CatBoostClassifier

print("="*50)
print("CATBOOST - OPTUNA")
print("="*50)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500, step=50),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'auto_class_weights': 'Balanced',
        'random_seed': 42,
        'verbose': 0
    }

    cat_model = CatBoostClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(cat_model, X_train, y_train, cv=skf, scoring='average_precision')
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest params: {study.best_params}")
print(f"Best CV PR-AUC: {study.best_value:.4f}")

best_cat = CatBoostClassifier(**study.best_params, auto_class_weights='Balanced', random_seed=42, verbose=0)
best_cat.fit(X_train, y_train)

y_proba = best_cat.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
print(f"Test PR-AUC: {average_precision_score(y_test, y_proba):.4f}")

# Threshold tuning
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred = (y_proba >= thresh).astype(int)
    if y_pred.sum() > 0:
        print(f"Threshold {thresh}: P={precision_score(y_test, y_pred):.3f}, R={recall_score(y_test, y_pred):.3f}, F1={f1_score(y_test, y_pred):.3f}")

importances = pd.DataFrame({'feature': X.columns, 'importance': best_cat.feature_importances_})
print(f"\nTop 10 features:\n{importances.sort_values('importance', ascending=False).head(10)}")

In [ ]:
y_proba_cat = best_cat.predict_proba(X_test)[:, 1]

for thresh in [0.2, 0.25, 0.3, 0.35, 0.4]:
    y_pred = (y_proba_cat >= thresh).astype(int)
    if y_pred.sum() > 0:
        print(f"Threshold {thresh}: P={precision_score(y_test, y_pred):.3f}, R={recall_score(y_test, y_pred):.3f}, F1={f1_score(y_test, y_pred):.3f}")

Let's do Ensemble

In [ ]:
print("="*50)
print("ENSEMBLE: XGBoost + CatBoost")
print("="*50)

# averaging probabilities
y_proba_ensemble = (y_proba + y_proba_cat) / 2

# Threshold tuning
print("\nThreshold tuning:")
for thresh in [0.3, 0.32, 0.35, 0.38, 0.4, 0.45]:
    y_pred = (y_proba_ensemble >= thresh).astype(int)
    if y_pred.sum() > 0:
        p = precision_score(y_test, y_pred)
        r = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        print(f"Threshold {thresh}: P={p:.3f}, R={r:.3f}, F1={f1:.3f}")

# Best threshold (based on F1)
best_thresh = 0.35
y_pred_ensemble = (y_proba_ensemble >= best_thresh).astype(int)

print("\n" + "="*50)
print("ENSEMBLE FINAL RESULTS")
print("="*50)
print(f"Threshold: {best_thresh}")
print(f"Precision: {precision_score(y_test, y_pred_ensemble):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_ensemble):.3f}")
print(f"F1: {f1_score(y_test, y_pred_ensemble):.3f}")
print(f"PR-AUC: {average_precision_score(y_test, y_proba_ensemble):.4f}")


In [ ]:
# comparison
results_df = pd.DataFrame({
    'Model': ['XGBoost', 'CatBoost', 'Ensemble'],
    'Threshold': [0.35, 0.3, 0.35],
    'Precision': [0.450, 0.714, 0.714],
    'Recall': [0.692, 0.385, 0.385],
    'F1': [0.545, 0.500, 0.500],
    'PR-AUC': [0.466, 0.405, 0.405]
})

print("="*60)
print("MODELS COMPARISON")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

# Best by F1
best_model = results_df.loc[results_df['F1'].idxmax(), 'Model']
print(f"\nBest model by F1: {best_model}")

# Best by Recall
best_recall = results_df.loc[results_df['Recall'].idxmax(), 'Model']
print(f"Best model by Recall: {best_recall}")

# Best by Precision
best_precision = results_df.loc[results_df['Precision'].idxmax(), 'Model']
print(f"Best model by Precision: {best_precision}")

So the conclusions are:


*   Best model by F1: XGBoost
*   Best model by Recall: XGBoost
*   Best model by Precision: CatBoost


If F1 or Recall is our primary concern, then XGBoost is the better choice.

If precision is the focus, then CatBoost should be selected